# NYC Taxi Data Cleaning
Loads 4 raw CSV files, aligns columns, cleans, and saves a valid dataset.

In [ ]:
import pandas as pd
import os

RAW_FILES = [
    '../Data/raw/yellow_tripdata_2015-01.csv',
    '../Data/raw/yellow_tripdata_2016-01.csv',
    '../Data/raw/yellow_tripdata_2016-02.csv',
    '../Data/raw/yellow_tripdata_2016-03.csv',
]
OUTPUT_PATH = '../Data/cleaned/cleaned_taxi_data.csv'

In [ ]:
# Canonical column name mapping — normalise before concat
COLUMN_RENAMES = {
    'RateCodeID': 'RatecodeID',   # 2015 uses RateCodeID, 2016 uses RatecodeID
    'rate_code':  'RatecodeID',
}

frames = []
for path in RAW_FILES:
    tmp = pd.read_csv(path, low_memory=False)
    # Strip whitespace from column names (common hidden issue)
    tmp.columns = tmp.columns.str.strip()
    # Normalise known column name variants
    tmp.rename(columns=COLUMN_RENAMES, inplace=True)
    print(f'Loaded {os.path.basename(path)}: {tmp.shape}')
    frames.append(tmp)

# Align all frames to a common column set before concat
all_cols = sorted(set(col for f in frames for col in f.columns))
frames = [f.reindex(columns=all_cols) for f in frames]

df = pd.concat(frames, ignore_index=True)
print('Step 1 (after concat):', df.shape)

In [ ]:
df['tpep_pickup_datetime']  = pd.to_datetime(df['tpep_pickup_datetime'],  errors='coerce')
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'], errors='coerce')

nat_count = df['tpep_pickup_datetime'].isna().sum() + df['tpep_dropoff_datetime'].isna().sum()
print(f'Step 2 (datetime conversion): {df.shape} | NaT values: {nat_count}')

In [ ]:
df['trip_duration'] = (
    df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
).dt.total_seconds() / 60  # minutes

null_duration = df['trip_duration'].isna().sum()
print(f'Step 3 (trip_duration computed): {df.shape} | NaN durations: {null_duration}')

In [ ]:
CRITICAL_COLS = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'trip_distance',
    'fare_amount',
    'trip_duration',
]

before = df.shape[0]
df.dropna(subset=CRITICAL_COLS, inplace=True)
print(f'Step 4 (selective dropna): {df.shape} | Rows dropped: {before - df.shape[0]}')

In [ ]:
before = df.shape[0]

mask = (
    (df['trip_distance'] > 0)    & (df['trip_distance'] < 50)   &
    (df['fare_amount']   > 0)    & (df['fare_amount']   < 500)  &
    (df['trip_duration'] > 0)    & (df['trip_duration'] < 120)
)

df = df[mask]
print(f'Step 5 (filtering): {df.shape} | Rows removed: {before - df.shape[0]}')

if df.shape[0] == 0:
    print('WARNING: All rows were filtered out. Check your filter bounds or upstream data.')

In [ ]:
df['pickup_hour']    = df['tpep_pickup_datetime'].dt.hour
df['pickup_weekday'] = df['tpep_pickup_datetime'].dt.dayofweek
df['pickup_month']   = df['tpep_pickup_datetime'].dt.month

print(f'Step 6 (feature engineering): {df.shape}')

In [ ]:
if df.shape[0] == 0:
    print('ERROR: DataFrame is empty. Skipping save.')
else:
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    df.to_csv(OUTPUT_PATH, index=False)
    print(f'Saved {df.shape[0]:,} rows to {OUTPUT_PATH}')

## Export Frontend Datasets
Loads the cleaned CSV and generates 6 aggregated files ready for the React dashboard.

In [ ]:
import pandas as pd
import os

# Load the cleaned dataset (self-contained — works even if run independently)
CLEANED_PATH = '../Data/cleaned/cleaned_taxi_data.csv'
OUT_DIR = '../Data/frontend'

df = pd.read_csv(CLEANED_PATH, low_memory=False)

# Ensure datetime columns are parsed
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'], errors='coerce')

# Derive columns needed for aggregations
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df['day_of_week'] = df['tpep_pickup_datetime'].dt.day_name()

os.makedirs(OUT_DIR, exist_ok=True)
print(f'Loaded {df.shape[0]:,} rows from {CLEANED_PATH}')

# ── 1. trips_by_hour ─────────────────────────────────────────────────────────
trips_by_hour = (
    df.groupby('pickup_hour', sort=True)
    .size()
    .reset_index(name='trip_count')
)
trips_by_hour.to_csv(f'{OUT_DIR}/trips_by_hour.csv', index=False)
print(f'✓ trips_by_hour.csv       → {len(trips_by_hour)} rows')

# ── 2. revenue_by_hour ───────────────────────────────────────────────────────
revenue_by_hour = (
    df.groupby('pickup_hour', sort=True)['total_amount']
    .sum()
    .round(2)
    .reset_index(name='total_revenue')
)
revenue_by_hour.to_csv(f'{OUT_DIR}/revenue_by_hour.csv', index=False)
print(f'✓ revenue_by_hour.csv     → {len(revenue_by_hour)} rows')

# ── 3. trips_by_day ──────────────────────────────────────────────────────────
DAY_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
trips_by_day = (
    df.groupby('day_of_week', sort=False)
    .size()
    .reset_index(name='trip_count')
)
trips_by_day['day_of_week'] = pd.Categorical(trips_by_day['day_of_week'], categories=DAY_ORDER, ordered=True)
trips_by_day = trips_by_day.sort_values('day_of_week').reset_index(drop=True)
trips_by_day.to_csv(f'{OUT_DIR}/trips_by_day.csv', index=False)
print(f'✓ trips_by_day.csv        → {len(trips_by_day)} rows')

# ── 4. distance_fare ─────────────────────────────────────────────────────────
distance_fare = (
    df.assign(distance_bucket=df['trip_distance'].round())
    .groupby('distance_bucket', sort=True)['fare_amount']
    .mean()
    .round(2)
    .reset_index(name='avg_fare')
    .rename(columns={'distance_bucket': 'distance_miles'})
)
distance_fare.to_csv(f'{OUT_DIR}/distance_fare.csv', index=False)
print(f'✓ distance_fare.csv       → {len(distance_fare)} rows')

# ── 5. tips_dist ─────────────────────────────────────────────────────────────
def tip_category(tip):
    if tip == 0:
        return 'No Tip'
    elif tip < 2:
        return 'Low (<$2)'
    elif tip < 5:
        return 'Medium ($2-$5)'
    else:
        return 'High ($5+)'

CAT_ORDER = ['No Tip', 'Low (<$2)', 'Medium ($2-$5)', 'High ($5+)']
tips_dist = (
    df['tip_amount']
    .apply(tip_category)
    .value_counts()
    .reset_index()
)
tips_dist.columns = ['tip_category', 'trip_count']
tips_dist['tip_category'] = pd.Categorical(tips_dist['tip_category'], categories=CAT_ORDER, ordered=True)
tips_dist = tips_dist.sort_values('tip_category').reset_index(drop=True)
tips_dist.to_csv(f'{OUT_DIR}/tips_dist.csv', index=False)
print(f'✓ tips_dist.csv           → {len(tips_dist)} rows')

# ── 6. map_data ──────────────────────────────────────────────────────────────
map_cols = ['pickup_latitude', 'pickup_longitude']
valid_map = df[map_cols].dropna()
nyc_mask = (
    valid_map['pickup_latitude'].between(40.4, 41.0) &
    valid_map['pickup_longitude'].between(-74.3, -73.6)
)
map_data = (
    valid_map[nyc_mask]
    .sample(n=min(50_000, int(nyc_mask.sum())), random_state=42)
    .reset_index(drop=True)
)
map_data.to_csv(f'{OUT_DIR}/map_data.csv', index=False)
print(f'✓ map_data.csv            → {len(map_data)} rows')

print(f'\nAll files saved to {OUT_DIR}/')